![adaptive-RAG flow](./adaptive-rag.png)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph


class AgentState(TypedDict):
    query: str
    context: list
    answer: str


graph_builder = StateGraph(AgentState)

In [ ]:
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results=3,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
)


def web_search(state: AgentState):
    query = state["query"]
    results = tavily_search_tool.invoke(query)
    print(f"web_search result == {results}")
    return {"context": results}

In [ ]:
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

client = Client()
generate_prompt = client.pull_prompt(
    "rlm/rag-prompt", dangerously_pull_public_prompt=True
)
generate_llm = ChatOpenAI(model="gpt-4o")


def web_generate(state: AgentState):
    context = state["context"]
    query = state["query"]
    rag_chain = generate_prompt | generate_llm | StrOutputParser()
    response = rag_chain.invoke({"question": query, "context": context})
    return {"answer": response}

In [ ]:
basic_llm = ChatOpenAI(model="gpt-4o-mini")


def basic_generate(state: AgentState):
    query = state["query"]
    basic_llm_chain = basic_llm | StrOutputParser()
    llm_response = basic_llm_chain.invoke(query)
    return {"answer": llm_response}

In [ ]:
# PromptTemplate 써도 되고 ChatPromptTemplate 써도 되고.
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal


class Route(BaseModel):
    target: Literal["vector_store", "llm", "web_search"] = Field(
        description="The target for the query to answer"
    )


router_system_prompt = """
You are an expert at routing a user's question to 'vector_store', 'llm', or 'web_search'.
'vector_store' contains information about income tax up to April 2026.
If you think the question is simple enough, use 'llm'.
If you think you need to search on web to answer the question, use 'web_search'.
"""

router_prompt = ChatPromptTemplate.from_messages(
    [("system", router_system_prompt), ("user", "{query}")]
)

router_llm = ChatOpenAI(model="gpt-4o-mini")
structured_router_llm = router_llm.with_structured_output(Route)


# expected answer : 'vector_store', 'llm' or 'web_search'
def router(state: AgentState):
    query = state["query"]
    router_chain = router_prompt | structured_router_llm
    route = router_chain.invoke({"query": query})
    print(f"router route == {route}")
    print(f"route.target == {route.target}")
    return route.target

In [ ]:
from income_tax_graph import graph as income_tax_subgraph

graph_builder.add_node("income_tax_agent", income_tax_subgraph)
graph_builder.add_node("web_search", web_search)
graph_builder.add_node("web_generate", web_generate)
graph_builder.add_node("basic_generate", basic_generate)

In [ ]:
from langgraph.graph import START, END

graph_builder.add_conditional_edges(
    START,
    router,
    {
        "vector_store": "income_tax_agent",
        "llm": "basic_generate",
        "web_search": "web_search",
    },
)

graph_builder.add_edge("web_search", "web_generate")
graph_builder.add_edge("web_generate", END)
graph_builder.add_edge("basic_generate", END)
graph_builder.add_edge("income_tax_agent", END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# test (target : vector_store)
initial_staet = {"query": "거주자의 연봉이 5천만원일 때 소득세는 얼마인가요?"}
graph.invoke(initial_staet)

In [ ]:
# test (target : web_search)
initial_staet = {"query": "여의도 맛집을 추천해주세요"}
graph.invoke(initial_staet)

In [ ]:
# test (target : basic_llm)
initial_staet = {"query": "What is the capital of Korea"}
graph.invoke(initial_staet)

# Adaptive RAG

## Overview

Adaptive RAG **dynamically selects the optimal answer path** by analyzing the user's question. Unlike traditional RAG which applies the same pipeline to every query, it routes questions to different nodes based on their nature, improving both efficiency and answer quality.

## Core Idea: Router

The core of this notebook is the **Router**. It uses an LLM with structured output to classify queries and route them to one of three paths:

| Route | Condition | How it's handled |
|-------|-----------|-----------------|
| `vector_store` | Income tax related questions | Calls the pre-built `income_tax_graph` as a subgraph |
| `web_search` | Questions requiring real-time information | Tavily search → generate answer via RAG prompt |
| `llm` | Simple, general knowledge questions | Directly answers using a smaller model (gpt-4o-mini) |

## Why Adaptive?

- **Cost efficiency**: No need to invoke search or large models for simple questions.
- **Better accuracy**: Domain-specific questions are routed to a dedicated subgraph for precise handling.
- **Extensibility**: Adding a new domain or tool only requires adding a new route to the router.

## Architecture

```
START
  │
  ├─ router (classify query)
  │     │
  │     ├── "vector_store" → income_tax_agent (subgraph) → END
  │     ├── "web_search"   → web_search → web_generate → END
  │     └── "llm"          → basic_generate → END
```

## Key Takeaways

1. **Router = LLM + Structured Output**. A Pydantic model (`Route`) enforces the output schema, ensuring reliable branching.
2. **Subgraph reuse**: The pre-built `income_tax_graph` is used as a single node. This is possible thanks to LangGraph's composable architecture.
3. **Conditional Edges**: `add_conditional_edges` determines the next node based on the router function's return value.